In [1]:
import pandas as pd
import numpy as np

In [2]:
datasets = {}
for i in range(1, 5):
    filename = f'Ookla_20_Q{i}.csv'

    df = pd.read_csv(filename)
    datasets[f'quarter_{i}'] = df
    print(f"Loaded {filename}: {len(df)} rows")

Loaded Ookla_20_Q1.csv: 4466078 rows
Loaded Ookla_20_Q2.csv: 4678047 rows
Loaded Ookla_20_Q3.csv: 4510196 rows
Loaded Ookla_20_Q4.csv: 3768743 rows


In [3]:
combined_data = []
for quarter_num, df in datasets.items():
    quarter_value = int(quarter_num.split('_')[1])
    df_copy = df.copy()
    df_copy['quarter'] = quarter_value
    df_copy['year'] = 2020
    combined_data.append(df_copy)

In [4]:
aggregated_df = pd.concat(combined_data, ignore_index=True)

In [5]:
aggregated_df.shape

(17423064, 9)

In [6]:
ookla_aggregated = aggregated_df.groupby(['quadkey'], as_index=False).agg({
    'avg_d_kbps': 'mean', 
    'avg_u_kbps': 'mean',
    'avg_lat_ms': 'mean',
    'tests': 'mean',
    'devices': 'mean',
    'quarter': 'count',
    'year': 'first',
    'geometry': 'first'
})

ookla_aggregated.rename(columns={'quarter': 'observations'}, inplace=True)

In [7]:
ookla_aggregated.shape

(7620215, 9)

In [8]:
ookla_aggregated.head()

,quadkey,avg_d_kbps,avg_u_kbps,avg_lat_ms,tests,devices,observations,year,geometry
0,22133222312322,5227.0,2449.0,54.0,2.0,2.0,1,2020,"POLYGON ((-160.02685546875 70.6435894914449, -..."
1,22133222313202,8630.0,3638.0,44.0,1.0,1.0,1,2020,"POLYGON ((-160.0048828125 70.6472305085909, -1..."
2,22133222330010,9529.0,3214.0,44.0,1.0,1.0,1,2020,"POLYGON ((-160.037841796875 70.6417687358462, ..."
3,22133222330011,9392.0,3390.0,42.0,1.0,1.0,1,2020,"POLYGON ((-160.032348632812 70.6417687358462, ..."
4,22133222330013,961.0,723.0,50.0,2.0,1.0,1,2020,"POLYGON ((-160.032348632812 70.6399478155463, ..."


In [9]:
ookla_aggregated.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7620215 entries, 0 to 7620214
Data columns (total 9 columns):
 #   Column        Dtype  
---  ------        -----  
 0   quadkey       int64  
 1   avg_d_kbps    float64
 2   avg_u_kbps    float64
 3   avg_lat_ms    float64
 4   tests         float64
 5   devices       float64
 6   observations  int64  
 7   year          int64  
 8   geometry      object 
dtypes: float64(5), int64(3), object(1)
memory usage: 523.2+ MB


In [10]:
ookla_aggregated.isnull().sum()

quadkey         0
avg_d_kbps      0
avg_u_kbps      0
avg_lat_ms      0
tests           0
devices         0
observations    0
year            0
geometry        0
dtype: int64

In [11]:
observation_counts = ookla_aggregated['observations'].value_counts().sort_index()

for obs_count, count in observation_counts.items():
    print(f"Quadkeys with {obs_count} observations: {count}")

Quadkeys with 1 observations: 3147473
Quadkeys with 2 observations: 1296673
Quadkeys with 3 observations: 1022031
Quadkeys with 4 observations: 2154038


In [12]:
grouped_agg = ookla_aggregated.groupby(['quadkey', 'observations'],  as_index=False).agg({
    'avg_d_kbps': 'mean',
    'avg_u_kbps': 'mean',  
    'avg_lat_ms': 'mean',
    'tests': 'mean',
    'devices': 'mean',
    'geometry': 'first'
})

grouped_agg.head()

,quadkey,observations,avg_d_kbps,avg_u_kbps,avg_lat_ms,tests,devices,geometry
0,22133222312322,1,5227.0,2449.0,54.0,2.0,2.0,"POLYGON ((-160.02685546875 70.6435894914449, -..."
1,22133222313202,1,8630.0,3638.0,44.0,1.0,1.0,"POLYGON ((-160.0048828125 70.6472305085909, -1..."
2,22133222330010,1,9529.0,3214.0,44.0,1.0,1.0,"POLYGON ((-160.037841796875 70.6417687358462, ..."
3,22133222330011,1,9392.0,3390.0,42.0,1.0,1.0,"POLYGON ((-160.032348632812 70.6417687358462, ..."
4,22133222330013,1,961.0,723.0,50.0,2.0,1.0,"POLYGON ((-160.032348632812 70.6399478155463, ..."


In [13]:
df = grouped_agg.copy()

cols_to_scale = ['avg_d_kbps', 'avg_u_kbps', 'avg_lat_ms', 'tests', 'devices']

for col in cols_to_scale:
    df[col] = df[col] / df['observations']

df['tests'] = df['tests'].round().astype(int)
df['devices'] = df['devices'].round().astype(int)

df = df.drop(columns='observations')
df.head()

,quadkey,avg_d_kbps,avg_u_kbps,avg_lat_ms,tests,devices,geometry
0,22133222312322,5227.0,2449.0,54.0,2,2,"POLYGON ((-160.02685546875 70.6435894914449, -..."
1,22133222313202,8630.0,3638.0,44.0,1,1,"POLYGON ((-160.0048828125 70.6472305085909, -1..."
2,22133222330010,9529.0,3214.0,44.0,1,1,"POLYGON ((-160.037841796875 70.6417687358462, ..."
3,22133222330011,9392.0,3390.0,42.0,1,1,"POLYGON ((-160.032348632812 70.6417687358462, ..."
4,22133222330013,961.0,723.0,50.0,2,1,"POLYGON ((-160.032348632812 70.6399478155463, ..."


In [14]:
df.describe()

,quadkey,avg_d_kbps,avg_u_kbps,avg_lat_ms,tests,devices
count,7.620215e+06,7.620215e+06,7.620215e+06,7.620215e+06,7.620215e+06,7.620215e+06
mean,9.136924e+14,3.507644e+04,1.710477e+04,2.916484e+01,3.453921e+00,1.083820e+00
std,6.827491e+14,5.700242e+04,3.823974e+04,1.132040e+02,8.681726e+00,1.714113e+00
min,2.213322e+13,1.000000e+00,7.500000e-01,0.000000e+00,0.000000e+00,0.000000e+00
25%,3.131032e+14,7.075750e+03,2.117062e+03,4.062500e+00,1.000000e+00,1.000000e+00
50%,1.202012e+15,1.785989e+04,5.710312e+03,8.666667e+00,1.000000e+00,1.000000e+00
75%,1.221131e+15,3.899303e+04,1.655725e+04,2.000000e+01,3.000000e+00,1.000000e+00
max,3.303302e+15,9.453850e+05,9.456040e+05,6.008000e+04,2.256000e+03,3.220000e+02


In [15]:
df.to_csv('Ookla_20.csv', index=False)